# Homework 5: Deploying Machine Learning Models

## Setup

In [2]:
import hashlib
import json
import pickle
import re
import subprocess
import tomllib

import requests
import sklearn

EXPECTED_SHA = '1646bbdcd38d4f044da6b630c5b332c93a314245a8c21929011c42de51f629f1'

with open('homework/05-deployment/pipeline.bin', 'rb') as f_in:
    sha = hashlib.sha256(f_in.read()).hexdigest()

print('sha256:', sha)
print('checksum OK:', sha == EXPECTED_SHA)

sha256: 1646bbdcd38d4f044da6b630c5b332c93a314245a8c21929011c42de51f629f1
checksum OK: True


## Q1. uv version

In [3]:
print(subprocess.run(['uv', '--version'], capture_output=True, text=True).stdout.strip())

uv 0.12.19 (Homebrew 2026-09-24 aarch64-apple-darwin)


## Q2. Scikit-Learn version

In [11]:
with open('homework/05-deployment/uv.lock', 'rb') as f:
    lock = tomllib.load(f)

sk_lock = [p['version'] for p in lock['package'] if p['name'] == 'scikit-learn']
print('uv.lock:', sk_lock)
print('installed:', sklearn.__version__)

with open('homework/05-deployment/pyproject.toml', 'rb') as f:
    deps = tomllib.load(f)['project']['dependencies']
print('pyproject:', [d for d in deps if 'scikit-learn' in d])

uv.lock: ['1.7.2']
installed: 1.7.2
pyproject: ['scikit-learn==1.7.2']


## Q3. Conversion probability (loaded model)

In [12]:
with open('homework/05-deployment/pipeline.bin', 'rb') as f_in:
    pipeline = pickle.load(f_in)

print(type(pipeline))

lead_q3 = {
    "lead_source": "paid_ads",
    "industry": "technology",
    "employment_status": "employed",
    "location": "north_america",
    "number_of_courses_viewed": 2,
    "annual_income": 79276.0,
    "interaction_count": 4,
    "lead_score": 0.41
}

def predict_single(obj, record):
    # sklearn Pipeline (DictVectorizer + LogisticRegression)
    if hasattr(obj, 'predict_proba'):
        return obj.predict_proba([record])[0, 1]
    # на случай, если сохранён кортеж (dv, model)
    dv, model = obj
    X = dv.transform([record])
    return model.predict_proba(X)[0, 1]

prob_q3 = predict_single(pipeline, lead_q3)
print(round(prob_q3, 3))

<class 'sklearn.pipeline.Pipeline'>
0.533


## Q4. Conversion probability (API)

In [13]:
url = 'http://localhost:9696'

lead_q4 = {
    "lead_source": "organic_search",
    "industry": "technology",
    "employment_status": "employed",
    "location": "europe",
    "number_of_courses_viewed": 4,
    "annual_income": 80304.0,
    "interaction_count": 7,
    "lead_score": 0.74
}

print(requests.get(f'{url}/health').json())

resp = requests.post(f'{url}/predict', json=lead_q4).json()
print(resp)

assert 0 <= resp['conversion_probability'] <= 1
assert isinstance(resp['conversion'], bool)
print(round(resp['conversion_probability'], 3))

# контроль: API должен совпасть с локальной моделью
print('local:', round(predict_single(pipeline, lead_q4), 3))

{'status': 'ok', 'model_sha256': '1646bbdcd38d4f044da6b630c5b332c93a314245a8c21929011c42de51f629f1', 'feature_defaults_sha256': '1694f842b908fac9f53c0cc19be6d91f934befc63af23cc32fefafde69acd0cd'}
{'conversion_probability': 0.769799, 'conversion': True}
0.77
local: 0.77


## Q5. Python base image

In [14]:
with open('homework/05-deployment/Dockerfile') as f:
    print(re.findall(r'^FROM\s+(\S+)', f.read(), flags=re.M))

['python:3.11.15-slim-bookworm@sha256:d29f48a31a8b408ed19272ca1e7b10ebae13b240a27e862d3d4217c528e2e0c3']


## Q6. Conversion probability (container)

In [16]:
health = requests.get(f'{url}/health').json()
print(health)

with open('homework/05-deployment/model_metadata.json') as f:
    meta = json.load(f)
print('metadata:', meta)

resp_c = requests.post(f'{url}/predict', json=lead_q4).json()
print(resp_c)
print(round(resp_c['conversion_probability'], 3))

{'status': 'ok', 'model_sha256': '1646bbdcd38d4f044da6b630c5b332c93a314245a8c21929011c42de51f629f1', 'feature_defaults_sha256': '1694f842b908fac9f53c0cc19be6d91f934befc63af23cc32fefafde69acd0cd'}
metadata: {'dataset': 'course_lead_scoring_2026.csv', 'dataset_sha256': '2d9da196bdefd2a45aa0a17ca21c19f06a5d9d56af628be1f04bbb3531cc12e9', 'rows': 5000, 'features': ['lead_source', 'industry', 'employment_status', 'location', 'number_of_courses_viewed', 'annual_income', 'interaction_count', 'lead_score'], 'target': 'converted', 'model': 'DictVectorizer + LogisticRegression(solver=liblinear, C=1.0)', 'random_state': 42, 'scikit_learn': '1.7.2', 'numeric_imputation': 'training-set median', 'numeric_defaults': {'number_of_courses_viewed': 2.0, 'annual_income': 55051.0, 'interaction_count': 4.0, 'lead_score': 0.49}, 'model_sha256': '1646bbdcd38d4f044da6b630c5b332c93a314245a8c21929011c42de51f629f1', 'feature_defaults_sha256': '1694f842b908fac9f53c0cc19be6d91f934befc63af23cc32fefafde69acd0cd'}
{'co